In [ ]:
!pip install pymupdf 
!pip install contractions 
!pip install num2words
!pip install jiwer
import num2words as n2w
import contractions as ctr
import pandas as pd
import pymupdf
import re
import string
import jiwer


# 1. Extract Data from reference progress note (ASSUMED PDF) + Get Data from NurseGPT (assumed CSV file)


In [ ]:
# Get refernce text:
mapping_color_doc = pd.read_excel(r"")
ref_notes = pymupdf.open(r"")
hypo_notes = pd.read_csv(r"") #Assume NurseGPT's artifacts stored as csv/ xlsx

def reg_pattern_find(text, page_number):
    pattern_list = {
         "Reference Diagnoses": r"Diagnoses\s*:(.*?)(?=\n[A-Z][a-z]+\s*:|$)",
    }
    result = {}
    
    for field_name, pattern in pattern_list.items():
        match = re.search(pattern, text, re.DOTALL)
        if match:
            result[field_name] =  match.group(1) 
        else:
             result[field_name] = None
             print(f"Page {page_number+1} does not contain the pattern, please recheck this page content")
    return result 
             

# Extract content from PDFs -> Convert into DF

def note_extraction(note_doc, mapping_doc):
    rows = []
    for page_num, page in enumerate(note_doc, start = 1): #Loop through each progress note (1 note = 1 page)
        note_content = page.get_text() #Get ALL content fields (Diagnoses details, Date Created, Physicians ID,etc)
        tier = mapping_color_doc.loc[mapping_color_doc['Page Number'] == page_num, 'Tier'].values[0]
        
        field_results = reg_pattern_find(note_content, page_num)
        field_results["Page Number"] = page_num
        field_results['Color Tier'] = tier 
        rows.append(field_results)
        
        reference_note_df = pd.DataFrame(rows)
        
    return reference_note_df 


#Set DF display option
pd.set_option('display.max_colwidth', None)
ref_notes = note_extraction(ref_notes, mapping_color_doc)
ref_notes

# 2. Data Preprocessing Steps

In [ ]:
spoken_ref_note = pd.read_csv(r"", index_col = False)
nurse_gpt_note = pd.read_csv(r"", index_col = False)

In [ ]:

def contraction(text):
    if isinstance(text, str):
        return ctr.fix(text)
    else:
        return text

def convert_number_words(match):
    number_str = match.group(0)
    number_int = int(number_str)
    return n2w.num2words(number_int)



def preprocessing_pipeline(df, column):
    df[column] = df[column].str.lower()  # Decapitalization
    df[column] = df[column].str.replace('\n', ' ')  # Remove newline

    df[column] = df[column].apply(lambda x: re.sub(r'\d+', convert_number_words, x))  # Convert numbers to words (82 -> 'eighty-two')

    df[column] = df[column].str.replace('-', ' ')
    df[column] = df[column].str.replace(':', ' ')
    df[column] = df[column].str.replace('/', 'over')
    df[column] = df[column].str.replace(r"'s\b", '', regex=True)
    # Contraction
    df[column] = df[column].apply(contraction)

    # Removal of punctuation
    punc_table = str.maketrans("", "", string.punctuation)
    df[column] = df[column].str.translate(punc_table)

    # Removal of white space
    df[column] = df[column].str.replace(r'\s+', ' ', regex=True)  # Extra white space
    df[column] = df[column].str.strip()  # Trailing white space

    return df

spoken_note_cleaned = preprocessing_pipeline(spoken_ref_note, 'Reference Diagnoses')
nurseGPT_transcript_cleaned = preprocessing_pipeline(nurse_gpt_note, 'Hypothesis Diagnoses')

# Merging Data (Reference Spoken Notes + Hypothesis Notes)

In [ ]:
merged_df = pd.merge(spoken_note_cleaned, nurseGPT_transcript_cleaned, on = ['Page Number', 'Color Tier'], how = 'outer')
merged_df = merged_df.loc[:, ['Page Number', 'Reference Diagnoses', 'Hypothesis Diagnoses', 'Color Tier']]

# WER Score Output

In [ ]:
def wer_computation(df, ref_col, hyp_col):
    error_score_list =[]
    sub_error_list = []
    del_error_list = []
    insertions_error_list = []
    corr_count_list = []
    
    
    #Extract score
    for ref_text, hyp_text in zip(df[ref_col], df[hyp_col]):
        if pd.isna(ref_text) or pd.isna(hyp_text) is None:
            ref_text = ''
            hyp_text = ''   
        error_report = jiwer.process_words(ref_text, hyp_text)
        error_score = error_report.wer * 100
        sub_error = error_report.substitutions
        del_error = error_report.deletions
        insertions_error = error_report.insertions
        corr_count = error_report.hits

        # Append list
        error_score_list.append(error_score)
        sub_error_list.append(sub_error)
        del_error_list.append(del_error)
        insertions_error_list.append(insertions_error)
        corr_count_list.append(corr_count)

    new_cols = pd.DataFrame({
    'Error Score': error_score_list,
    'Substitution Error Count': sub_error_list,
    'Deletion Error Count': del_error_list,
    'Insertion Error Count': insertions_error_list,
    'Correct Transcribed Word Count': corr_count_list
    })

    df = pd.concat([df, new_cols], axis = 1)
    return df

df_wer_computed = wer_computation(merged_df, 'Reference Diagnoses', 'Hypothesis Diagnoses')
